In [1]:
import caesar
import yt
import numpy as np
from scipy.stats import binned_statistic
import matplotlib.pyplot as plt
import os
import unyt
from functools import partial



cuts_dict    = [{'field':('gas','density'), '<':(0.1*unyt.physical_constants.proton_mass)*unyt.cm**-3},
                    {'field':("gas", "temperature"), '>': 2e5*unyt.K},
                    {'field':("PartType0", 'Density'), '<':(0.1*unyt.physical_constants.proton_mass)*unyt.cm**-3},
                    {'field':("PartType0", "temperature"), '>': 2e5*unyt.K},
                    {'field':("PartType0", "DelayTime"), '=':0},
                    {'field':("PartType0", "StarFormationRate"), '=':0}
                    ]

In [2]:
def _filtered_gas(pfilter, data, dataset_cuts):
    pfilter = True
    for i, cut in enumerate(dataset_cuts):
        if cut.get("=",None) != None:
            pfilter &= data[cut["field"]] == cut["="]
        else:
            if cut.get(">",None) != None:
                pfilter &= data[cut["field"]] > cut[">"]
            if cut.get("<",None) != None:
                pfilter &= data[cut["field"]] < cut["<"]  
                        
    return pfilter


yt.add_particle_filter(
    "filtered_gas",
    function=partial(_filtered_gas, dataset_cuts = cuts_dict),
    filtered_type="PartType0",
    requires=['density', 'temperature', 'DelayTime', 'StarFormationRate'],
)



In [3]:
NUM_R500_FOR_PROJ = 3
CAESAR_HALO_IDX = 0

snap_file = '/scratch/b/babul/aspadawe/snapshots/Hyenasc_L0_zooms/halo_3224/snap_halo_3224_151.hdf5'
csr_obj = caesar.load('/scratch/b/babul/aspadawe/snapshots/Hyenasc_L0_zooms/halo_3224/Groups/Caesar_halo_3224_151.hdf5')

ds = yt.load(snap_file)

csr_massive_halo = csr_obj.halos[CAESAR_HALO_IDX]
halo_r500   = csr_massive_halo.virial_quantities["r500c"]
halo_m500   = csr_massive_halo.virial_quantities["m500c"]


# central_gal = [x for x in csr_obj.galaxies if x.GroupID == csr_massive_halo.central_galaxy][0]
central_gal_stellar_mass = csr_massive_halo.central_galaxy.masses['stellar']
        
print('%.2f' % float(np.log10(central_gal_stellar_mass.to("Msun").value)))
print('%.2f' % float(np.log10(halo_m500.to("Msun").value)))
        # exit()
halo_center = csr_massive_halo.minpotpos
halo_center_original = csr_massive_halo.minpotpos
ds_fid = yt.load(snap_file)
start_time = ds_fid.current_time.to("Myr").value
start_scale_factor = ds_fid.scale_factor
sp_fid = ds_fid.sphere(center = halo_center , radius = halo_r500)
SMBH_pid = int(sp_fid["PartType5", "particle_index"][np.argmax(sp_fid["PartType5", "BH_Mass"])])

ad = ds.all_data()
halo_center_post = ad["PartType5", "particle_position"][ad["PartType5", "particle_index"] == SMBH_pid][0]


sp  = ds.sphere(center = halo_center_post , radius = (NUM_R500_FOR_PROJ*halo_r500.to("kpc").value,'kpc'))


ds.add_particle_filter("filtered_gas",)


yt : [INFO     ] 2025-03-13 16:32:56,343 Opening /scratch/b/babul/aspadawe/snapshots/Hyenasc_L0_zooms/halo_3224/Groups/Caesar_halo_3224_151.hdf5
yt : [INFO     ] 2025-03-13 16:32:56,382 Found 1021 halos
yt : [INFO     ] 2025-03-13 16:32:56,388 Found 320 galaxies
yt : [INFO     ] 2025-03-13 16:32:57,775 ComovingIntegrationOn does not exist, falling back to OmegaLambda
yt : [INFO     ] 2025-03-13 16:32:57,789 Calculating time from 1.000e+00 to be 4.375e+17 seconds
yt : [INFO     ] 2025-03-13 16:32:57,790 Assuming length units are in kpc/h (comoving)
yt : [INFO     ] 2025-03-13 16:32:57,886 Parameters: current_time              = 4.3748565942961094e+17 s
yt : [INFO     ] 2025-03-13 16:32:57,887 Parameters: domain_dimensions         = [1 1 1]
yt : [INFO     ] 2025-03-13 16:32:57,888 Parameters: domain_left_edge          = [0. 0. 0.]
yt : [INFO     ] 2025-03-13 16:32:57,889 Parameters: domain_right_edge         = [200000. 200000. 200000.]
yt : [INFO     ] 2025-03-13 16:32:57,889 Parameters:

10.68
12.87


yt : [INFO     ] 2025-03-13 16:32:58,281 Allocating for 7.504e+06 particles
Loading particle index: 100%|██████████| 10/10 [00:00<00:00, 2524.41it/s]
yt : [WARNING  ] 2025-03-13 16:33:01,016 It appears that you are filtering on an SPH field type. It is recommended to use 'gas' as the filtered particle type in this case instead.


True

In [4]:
print("doing binned entropy")
entropy_radius_bins = np.linspace(-3,np.log10(NUM_R500_FOR_PROJ), 500)
binned_entropy_all_gas_medians, entropy_bin_edges, _ = binned_statistic(np.log10(sp["gas","radius"].to("kpc").value/halo_r500), (unyt.physical_constants.kb * sp["gas", "temperature"] / sp["gas", "El_number_density"]**(2/3)).to("keV*cm**2").value, bins = entropy_radius_bins)
        
binned_entropy_filtered_gas_medians, _, _ = binned_statistic(np.log10(sp["filtered_gas","radius"].to("kpc").value/halo_r500), (unyt.physical_constants.kb * sp["filtered_gas", "temperature"] / sp["filtered_gas", "El_number_density"]**(2/3)).to("keV*cm**2").value, bins = entropy_radius_bins)


doing binned entropy


In [7]:
entropy_fig = plt.figure(figsize=(6,4))
plt.plot( np.asarray(entropy_bin_edges[:-1]) + 0.5*np.diff(entropy_bin_edges)[0], np.log10(binned_entropy_filtered_gas_medians), )
plt.title("Entropy Profile Testing")
plt.xlabel("Log R/R500")
plt.ylabel("Log Entropy (keV cm^2)")
plt.xlim(-1, 0.5)
plt.ylim(2.5, 3.2)
plt.savefig('/home/b/babul/stlock/entropy_profiles/outputs/entropy_profile.png')
plt.close(entropy_fig)